In [78]:
from dotenv import load_dotenv

In [79]:
load_dotenv()

True

In [80]:
# set model settings control the model's parameters
from pydantic_ai.models.openrouter import OpenRouterModelSettings
from pydantic_ai.settings import ModelSettings
settings = OpenRouterModelSettings(
    temperature=0.0,
    max_tokens=10000,
    top_p=1.0, #（核采样）
    frequency_penalty=0.5, #（频率惩罚）
    presence_penalty=0.5,   #（存在惩罚）
)
zai_model_settings = ModelSettings(
    temperature=0.0,
    max_tokens=10000,
    top_p=1.0,
    frequency_penalty=0.5,
    presence_penalty=0.5,
)

In [87]:
#structure output and define output type
from pydantic_ai import Agent
from pydantic import BaseModel


class person(BaseModel):
    name:str
    age:int
    job:str

agent = Agent(
    'openrouter:xiaomi/mimo-v2-flash',
    instructions="""
    You are **Receptionist** decide which data sources to use based on the user's request.
    **Task Instructions**based on the user's query to answer, and summarize the output.
    **audience**：Support developer
    **tone**：clear and concise.
    **max_length**：≤ 120 words.
    """,
    model_settings=settings,
    output_type=person,
)
response = await agent.run("Evan is a 21 year old AI Agent developer in SUZHOU")
print(response.output)
response = await agent.run("Draven is a 22 year old hardware engineer in HANGZHOU")
print(response.output)

name='Evan' age=21 job='AI Agent developer'
name='Draven' age=22 job='hardware engineer'


In [24]:
@agent.tool_plain
def get_job(name: str) -> str:
    """Fuction that return the job base on person's name"""
    if name == 'Draven':
        return 'hardware engineer'
    elif name == 'Evan':
        return 'AI Agent developer'
    else:
        return 'No information found'


In [27]:
response = await agent.run("Whit is my job? my name is Evan")
print(response.output)

Your job is **AI Agent developer**.


In [ ]:
from pydantic_ai import RunContext

roulette_agent = Agent(
    'openrouter:xiaomi/mimo-v2-flash',
    deps_type=int,
    instructions="""
    You are **Roulette Agent** process number guesses from user.
    """,
    model_settings=settings,
    output_type=bool,
)

@roulette_agent.tool
async def guess_number(ctx:RunContext[int],guessed_number:int)->str:
    """Guess a random number for the user"""
    if guessed_number == ctx.deps:
        return 'win'
    else:
        return 'lose'

correct_number = 22
response = await roulette_agent.run('I want put my money on 22!',deps=correct_number)
print(response.output)


True


In [61]:
import sqlite3
from dataclasses import dataclass

@dataclass
class MyDependencies:
    user_name:str
    db_connection:sqlite3.Connection

agent = Agent(
    'openrouter:xiaomi/mimo-v2-flash',
    model_settings=settings,
    deps_type=MyDependencies,
    instructions="""
    You are **Database Agent** process user's request based on database.
    **Task Instructions** Always start every singal message by calling user's name.
    **Audience** Support developer
    **Tone** Clear and natural
    **Max Summary Length** ≤ 120 words
    """,
    output_type=list[tuple],

)

@agent.system_prompt
def get_user_name(ctx:RunContext[MyDependencies])->str:
    """Get user name from database"""
    return f'the user name is {ctx.deps.user_name}'

@agent.tool
def list_all_users_info(ctx:RunContext[MyDependencies])->list[tuple]:
    """List all users info from database"""
    connection = ctx.deps.db_connection
    cursor = connection.cursor()

    cursor.execute('SELECT name, age, job FROM users;')
    user_info = cursor.fetchall()

    result = []

    for row in user_info:
        result.append((row[0],row[1],row[2]))

    return result

db_connection = sqlite3.connect('test.db',check_same_thread=False)
deps = MyDependencies(user_name='Draven',db_connection=db_connection)

response = await agent.run('What is all users info?',deps=deps)
print(response.output)



[('Draven', 22, 'hardware engineer'), ('Evan', 21, 'AI Agent developer')]


In [ ]:
#make toolsset
import datetime as dt
from pydantic_ai.toolsets import FunctionToolset

def get_current_date()->str:
    """Get current date"""
    return dt.datetime.now().strftime('%Y-%m-%d')

def get_current_weekday()->str:
    """Get current weekday"""
    return dt.datetime.now().strftime('%A')

def get_current_time()->str:
    """Get current time"""
    return dt.datetime.now()

detetime_tools = FunctionToolset([get_current_date,get_current_weekday,get_current_time])


In [ ]:
#use toolsset in agent
from pydantic_ai import Agent
agent = Agent(
    'openrouter:xiaomi/mimo-v2-flash',
    model_settings=settings,
    instructions="""
    You are **Time Agent** process user's request based on time.
    """,
    toolsets=[detetime_tools],
)

response = await agent.run('What is the current time information?')
print(response.output)

Here's the current time information:

- **Date**: March 22, 2026  
- **Day of the Week**: Sunday  
- **Time**: 6:38 PM (18:38:11)  

Let me know if you need further details!


In [86]:
#tool_timeout: timeout for the tool
#retries: number of retries for the tool
import asyncio

agent = Agent(
    'openrouter:xiaomi/mimo-v2-flash',
    model_settings=settings,
    instructions="""
    You are helpful assistant who can answer user's query.
    """,
    tool_timeout=6,
    retries=3,
)

@agent.tool_plain
async def get_favorite_color() -> str:
    await asyncio.sleep(3)
    return 'blue'

response = await agent.run('What is my favorite color?')
print(response.output)


Your favorite color is blue! 🌊


In [88]:
#built-in tool
from pydantic_ai import WebSearchTool
from pydantic_ai import Agent

agent = Agent(
    'openrouter:xiaomi/mimo-v2-flash',
    model_settings=settings,
    instructions="""
    You are helpful Assistant who can answer user's query.
    **Task Instructions** give a summary of the output including details.
    **Audience** Support AI developer
    **Tone** Clear and natural

    **Example**
    User: what is SUZHOU foods native people like eat?
    Assistant: Suzhou people enjoy a variety of foods, with local favorites including Suzhou soup noodles, sweet and sour mandarin fish, and pan-fried buns. 
    details:
    - title: Top 10 Suzhou Foods You Should Try.
    - source: https://www.travelchinaguide.com/cityguides/jiangsu/suzhou/dining.html

    **Warning** always output in chinese.
    """,
    builtin_tools=[WebSearchTool()],
    )
response = await agent.run('what is the newest AI news?')
print(response.output)

根据2026年3月22日的最新网络搜索结果，以下是当前AI领域的最新动态：

**OpenAI推出全自动AI研究员计划**  
OpenAI正全力研发一种完全自动化的AI研究员系统，旨在让AI自主处理复杂问题。该计划分为两个阶段：  
1. **AI研究实习生**：预计2026年9月推出，可独立完成小型研究任务。  
2. **全自动多智能体系统**：计划2028年发布，能解决人类难以应对的大型问题（如数学证明、生命科学等）。  
OpenAI首席科学家Jakub Pachocki表示，这将成为公司未来几年的“北极星”目标，整合推理模型、智能体和可解释性技术。  
[technologyreview.com](https://www.technologyreview.com/2026/03/20/1134438/openai-is-throwing-everything-into-building-a-fully-automated-researcher/)  
[newsbytesapp.com](https://www.newsbytesapp.com/news/science/openai-targets-autonomous-ai-researcher-as-next-big-breakthrough/story)

**美国白宫发布AI监管框架**  
白宫于2026年3月20日公布国家AI立法框架，旨在统一各州法规并推动轻量级监管。核心内容包括：  
- 预防各州制定独立AI法律，支持联邦主导的行业特定监管。  
- 简化数据中心许可流程，打击AI诈骗，平衡知识产权与AI训练需求。  
- 强调“美国优先”创新，避免政府强制内容审查。  
该框架由特朗普政府推动，旨在加速AI发展并维护国家安全。  
[cnn.com](https://www.cnn.com/2026/03/20/tech/white-house-ai-framework)

**WordPress.com开放AI代理内容管理权限**  
WordPress.com于2026年3月20日升级其Model Context Protocol（MCP）服务器，允许AI代理（如Claude、ChatGPT）通过自然对话直接管理网站内容。新功能包括：  
- 起草、编辑和发布帖子/页面。  
- 

In [62]:
#data analysis assistant 
#use E2B execute_python to calculate the result
from pydantic_ai import Agent
from e2b_code_interpreter import Sandbox
import json
from langfuse import get_client
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
model = OpenAIChatModel(
    'glm-5-turbo',
    provider=OpenAIProvider(
        base_url='https://open.bigmodel.cn/api/coding/paas/v4', api_key='ae5c165b8b284ab3ba9d561b72fc228a.bt1xu0szPtLETStP'
    ),
)
langfuse = get_client()
Agent.instrument_all()

def execute_python(code: str) -> str:
    """
    Execute python code in a Jupyter notebook.
    """
    with Sandbox.create() as sandbox:
        execution = sandbox.run_code(code)
        stdout = [str(x) for x in execution.logs.stdout]
        result = {
            "text": execution.text,
            "stdout": stdout,
            "error": str(getattr(execution, "error", None)) if getattr(execution, "error", None) else None,
        }
        return json.dumps(result, ensure_ascii=False, default=str)


agent = Agent(
    model,
    model_settings=zai_model_settings,
    instructions="""
    You are a **Data analysis assistant**.
    **Task Instructions** calculate and analyze the result to exactly answer the user's question.
    **Audience** Support developer
    **Tone** Clear and natural
    **Structure output** human-readable
    """,
    tools=[execute_python],
)

response = await agent.run(
    """
假设你投资了三支股票：

A: 初始1000元，年增长率 5%
B: 初始1000元，年增长率 7%
C: 初始1000元，年增长率 4%

计算5年后的总资产，并判断哪支股票收益最高。
""",
)
print(response.output)


## 📈 5 年后各股票资产及总资产

| 股票 | 初始投资 | 年增长率 | 计算公式 | 5 年后资产 |
|------|---------|---------|---------|-----------|
| **A** | ¥1,000 | 5% | 1000 × (1.05)⁵ | **¥1,276.28** |
| **B** | ¥1,000 | 7% | 1000 × (1.07)⁵ | **¥1,402.55** |
| **C** | ¥1,000 | 4% | 1000 × (1.04)⁵ | **¥1,216.65** |

### 💰 合计

$$
\text{总资产} = 1{,}276.28 + 1{,}402.55 + 1{,}216.65 = \boxed{¥3{,}895.49}
$$

### 🏆 最高收益判断

- **股票 B（年增长率 7%）** 收益最高，5 年后资产为 ¥1,402.55  
- 相比初始投资，B 的收益率为 **40.26%**（¥402.55），远高于 A 的 27.63% 和 C 的 21.67%。

> **结论：在给定的条件下，选择股票 B 是最优策略。**  
> 若总投资额固定，可考虑将更多资金分配给 B 以最大化整体收益。


In [66]:
from pydantic_ai import Agent,Embedder
embedder = Embedder('openrouter:qwen/qwen3-embedding-8b')
embedding = await embedder.embed_query('What is the capital city of China?')
print(embedding)


EmbeddingResult(embeddings=[[-0.016746949404478073, 0.0366264209151268, -0.006084322929382324, -0.024457773193717003, -0.004969867877662182, -0.006024082191288471, 0.0036445697769522667, 0.011566237546503544, 0.0012048163916915655, 0.005963841453194618, -0.008614437654614449, -0.033493898808956146, -0.007469861768186092, -0.008313233032822609, 0.07373476773500443, 0.019879471510648727, -0.007530102971941233, -0.00554215582087636, 0.03132522851228714, -0.008192751556634903, 0.007379500661045313, 0.030240893363952637, 0.0031475829891860485, -0.019638508558273315, -0.007198778446763754, 0.016385503113269806, 0.01150599680840969, -0.001543671009130776, 0.019879471510648727, -0.004548182245343924, 0.012891535647213459, -0.04024086892604828, 0.010301180183887482, 0.005451794248074293, -0.0017093332717195153, 0.014879482798278332, -0.008975882083177567, 0.0005722878267988563, -0.009457808919250965, 0.006114443298429251, 0.025903552770614624, 0.014999964274466038, -0.019638508558273315, 0.0207

C:\Users\Draven\AppData\Local\Temp\ipykernel_29488\629642558.py:3: RuntimeWarning: coroutine 'Embedder.embed_query' was never awaited
  embedding = await embedder.embed_query('What is the capital city of China?')


In [67]:
embedding

EmbeddingResult(embeddings=[[-0.016746949404478073, 0.0366264209151268, -0.006084322929382324, -0.024457773193717003, -0.004969867877662182, -0.006024082191288471, 0.0036445697769522667, 0.011566237546503544, 0.0012048163916915655, 0.005963841453194618, -0.008614437654614449, -0.033493898808956146, -0.007469861768186092, -0.008313233032822609, 0.07373476773500443, 0.019879471510648727, -0.007530102971941233, -0.00554215582087636, 0.03132522851228714, -0.008192751556634903, 0.007379500661045313, 0.030240893363952637, 0.0031475829891860485, -0.019638508558273315, -0.007198778446763754, 0.016385503113269806, 0.01150599680840969, -0.001543671009130776, 0.019879471510648727, -0.004548182245343924, 0.012891535647213459, -0.04024086892604828, 0.010301180183887482, 0.005451794248074293, -0.0017093332717195153, 0.014879482798278332, -0.008975882083177567, 0.0005722878267988563, -0.009457808919250965, 0.006114443298429251, 0.025903552770614624, 0.014999964274466038, -0.019638508558273315, 0.0207

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.settings import ModelSettings

model = OpenAIChatModel(
    'glm-5-turbo',
    provider=OpenAIProvider(
        base_url='https://open.bigmodel.cn/api/coding/paas/v4', api_key='ae5c165b8b284ab3ba9d561b72fc228a.bt1xu0szPtLETStP'
    ),
)
zai_model_settings = ModelSettings(
    temperature=0.0,
    max_tokens=10000,
    top_p=1.0,
    frequency_penalty=0.5,
    presence_penalty=0.5,
)
agent = Agent(model,model_settings=zai_model_settings)
response = await agent.run('What is the capital city of China?')
print(response.output)

The capital city of China is Beijing.


In [63]:
response.all_messages()

[ModelRequest(parts=[UserPromptPart(content='\n假设你投资了三支股票：\n\nA: 初始1000元，年增长率 5%\nB: 初始1000元，年增长率 7%\nC: 初始1000元，年增长率 4%\n\n计算5年后的总资产，并判断哪支股票收益最高。\n', timestamp=datetime.datetime(2026, 3, 22, 16, 9, 12, 42658, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 3, 22, 16, 9, 12, 43106, tzinfo=datetime.timezone.utc), instructions="You are a **Data analysis assistant**.\n    **Task Instructions** calculate and analyze the result to exactly answer the user's question.\n    **Audience** Support developer\n    **Tone** Clear and natural\n    **Structure output** human-readable", run_id='ff582e47-6428-4b3a-896e-31eb65d3aede'),
 ModelResponse(parts=[ThinkingPart(content="The user wants to calculate the total asset after 5 years for three stocks with different growth rates, all with initial investment of 1000. Then compare and determine which has the highest returns.\n\nWe can calculate using compound interest formula: Future value = Principal * (1 + r)^n\n\n- A: 1000 * (1.05)^5

In [ ]:
response.output

'我来帮你计算这三支股票5年后的收益情况。让我用代码来精确计算并分析结果。\n\n```python\nimport pandas as pd\n\n# 初始投资数据\ninitial_investment = 1000\nstocks = {\n    \'A\': 0.05,  # 年增长率 5%\n    \'B\': 0.07,  # 年增长率 7%\n    \'C\': 0.04   # 年增长率 4%\n}\n\n# 计算每年的价值\nyears = range(1, 6)  # 1-5年\nresults = {}\n\nfor stock, growth_rate in stocks.items():\n    yearly_values = []\n    for year in years:\n        value = initial_investment * (1 + growth_rate) ** year\n        yearly_values.append(round(value, 2))\n    results[stock] = yearly_values\n\n# 创建DataFrame便于展示\ndf = pd.DataFrame(results, index=[f\'第{y}年\' for y in years])\n\n# 计算5年后的最终收益\nfinal_values = {stock: values[-1] for stock, values in results.items()}\nhighest_stock = max(final_values, key=final_values.get)\n\n# 输出结果\nprint("📊 三支股票5年投资分析")\nprint("=" * 50)\nprint("\\n📈 每年价值变化：")\nprint(df.to_string())\nprint(f"\\n💰 5年后的最终价值：")\nfor stock, value in final_values.items():\n    profit = value - initial_investment\n    print(f"  {stock}: {value:.2f}元 (收益: +{profit:.2f}

In [ ]:
import sqlite3

conn = sqlite3.connect('test.db',check_same_thread=False)
conn.execute("CREATE TABLE IF NOT EXISTS users (name TEXT, age INTEGER, job TEXT)")

conn.commit()

conn.close()


In [ ]:
#手动操作DB
import sqlite3

conn = sqlite3.connect('test.db',check_same_thread=False)

conn.execute("DELETE FROM users WHERE rowid = 2")
conn.commit()
conn.close()


In [83]:
#比较system_prompt和instructions的区别

from pydantic_ai import Agent
agent1 = Agent(
    'openrouter:arcee-ai/trinity-mini:free',
    system_prompt="""
        if user ask you what is your name, you should answer "I am a AI Agent1", otherwise you should answer the question normally.
    """,
    model_settings=settings,)
response = await agent1.run('What is capital city of china')
history = response.all_messages()
print(response.output)
agent2 = Agent(
    'openrouter:arcee-ai/trinity-mini:free',
    system_prompt="""
        if user ask you what is your name, you should answer "I am a AI Agent2", otherwise you should answer the question normally.
    """,
    model_settings=settings,)
response = await agent2.run('what is your name?',message_history=history)
print(response.output)



The capital city of China is Beijing.


I am a AI Agent1


In [84]:
response.all_messages()

[ModelRequest(parts=[SystemPromptPart(content='\n        if user ask you what is your name, you should answer "I am a AI Agent1", otherwise you should answer the question normally.\n    ', timestamp=datetime.datetime(2026, 3, 22, 17, 7, 21, 328441, tzinfo=datetime.timezone.utc)), UserPromptPart(content='What is capital city of china', timestamp=datetime.datetime(2026, 3, 22, 17, 7, 21, 328441, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 3, 22, 17, 7, 21, 328441, tzinfo=datetime.timezone.utc), run_id='0c9b314b-1c8d-402d-b066-d5722f5c7e3a'),
 ModelResponse(parts=[ThinkingPart(content='Okay, the user asked about the capital city of China. Let me check if they mentioned my name. The query is straightforward: "What is capital city of china". There\'s no mention of "AI Agent1" here. According to the instructions, if they don\'t ask about my name, I should answer normally. So I need to provide the correct answer. The capital of China is Beijing. I\'ll state that clearly

In [85]:
print(history)

[ModelRequest(parts=[SystemPromptPart(content='\n        if user ask you what is your name, you should answer "I am a AI Agent1", otherwise you should answer the question normally.\n    ', timestamp=datetime.datetime(2026, 3, 22, 17, 7, 21, 328441, tzinfo=datetime.timezone.utc)), UserPromptPart(content='What is capital city of china', timestamp=datetime.datetime(2026, 3, 22, 17, 7, 21, 328441, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 3, 22, 17, 7, 21, 328441, tzinfo=datetime.timezone.utc), run_id='0c9b314b-1c8d-402d-b066-d5722f5c7e3a'), ModelResponse(parts=[ThinkingPart(content='Okay, the user asked about the capital city of China. Let me check if they mentioned my name. The query is straightforward: "What is capital city of china". There\'s no mention of "AI Agent1" here. According to the instructions, if they don\'t ask about my name, I should answer normally. So I need to provide the correct answer. The capital of China is Beijing. I\'ll state that clearly 

In [ ]:
#await: wait for the result
response = await agent.run('Draven is a 22 year old hardware engineer in HANGZHOU')
print(response.output)



Draven is a 22-year-old hardware engineer based in Hangzhou. His expertise likely encompasses circuit design, prototyping, and testing. He may be active in professional networks like LinkedIn or local engineering communities. This summary uses publicly available professional data sources.


In [46]:
response = await agent.run('What did you say before?',message_history=response.new_messages())
print(response.output)



Ipreviously summarized Draven as a 22-year-old hardware engineer in Hangzhou, summarizing his profession, expertise, and potential professional networks based on publicly available data.


In [ ]:
#async output combine to behave like stream output
#run_stream:invoke stream output interface
#delta=True: combine the new added output in real time
#flush=True: output the result immediately
async with agent.run_stream('Evan is a 21 year old AI Agent developer in SUZHOU') as run:
    async for chunk in run.stream_text(delta=True):
        print(chunk,end='', flush=True)